In [21]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/comm_log.db")

campaign = pd.read_sql_query("SELECT * FROM campaign", conn)
comm_log = pd.read_sql_query("SELECT * FROM communication_log", conn)

Both the datasets: campaign, communication_log loaded from SQLite db into pandas

In [22]:
campaign

,id,merchant_id,parent_id,name,creation_status,processing_status
0,9001,501,NaN,Diwali Cart Recovery - Wave 1,approved,processed
1,9002,501,9001.0,Diwali Cart Recovery - Retry A,approved,processed
2,9003,501,9002.0,Diwali Cart Recovery - Retry B,approved,processed
3,9004,501,9001.0,Diwali Cart Recovery - Retry C (pending),approval_awaiting,processed
4,9101,501,NaN,Diwali Flash Sale - Standalone,approved,processed
5,9201,501,NaN,Diwali Wave 2,approved,processed
6,9202,501,9201.0,Diwali Wave 2 - Retry,approved,processed


campaign has 7 rows
- observation: only 9004 id has approval awaiting

In [23]:
comm_log

,id,merchant_id,communication_id,customer_id,communication_type,delivery_status,sent_time,scheduled_time,credit_used,channel
0,1,501,9001,C1,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
1,2,501,9001,C2,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
2,3,501,9002,C2,2,900,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
3,4,501,9001,C3,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
4,5,501,9002,C3,2,1100,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
5,6,501,9003,C3,2,900,2026-10-05 10:00:00,2026-10-05 10:00:00,1,sms
6,7,501,9001,C4,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
7,8,501,9001,C5,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
8,9,501,9001,C6,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
9,10,501,9001,C7,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms


communication log has 30 rows, all are the attempts made to send sms

- observation: some attempts are made to the same customer, meaning they failed in the first campaign, and retired in the next one
- 1100 means a failed attempt
- 900 means a successful attemt

In [ ]:
## naive count:
query = """
SELECT COUNT(*) as naive_count
FROM communication_log
WHERE merchant_id = 501
"""
pd.read_sql_query(query, conn)

,naive_count
0,30


Counting every row for merchant 501: giving 30

In [40]:
## checking campaign eligibility
query = """
SELECT id, creation_status, processing_status
FROM campaign
WHERE merchant_id = 501
"""
pd.read_sql_query(query, conn)


,id,creation_status,processing_status
0,9001,approved,processed
1,9002,approved,processed
2,9003,approved,processed
3,9004,approval_awaiting,processed
4,9101,approved,processed
5,9201,approved,processed
6,9202,approved,processed


Looking at `creation_status` and `processing_status` for all 7 campaigns: everything is approved + processed except campaign 9004, which is stuck at `approval_awaiting`. Per the data dictionary, a campaign only counts toward reporting once its creation status is finalized — 9004 hasn't cleared that yet, even though sends already went out under it. This is a edge-case in the pipeline

In [ ]:
query = """
SELECT * from communication_log
WHERE communication_id = 9004
"""
pd.read_sql_query(query, conn)

,id,merchant_id,communication_id,customer_id,communication_type,delivery_status,sent_time,scheduled_time,credit_used,channel
0,14,501,9004,C11,2,900,2026-10-06 10:00:00,2026-10-06 10:00:00,1,sms
1,15,501,9004,C12,2,900,2026-10-06 10:00:00,2026-10-06 10:00:00,1,sms
2,16,501,9004,C13,2,900,2026-10-06 10:00:00,2026-10-06 10:00:00,1,sms
3,17,501,9004,C14,2,900,2026-10-06 10:00:00,2026-10-06 10:00:00,1,sms


we see 4 communication logs that have the communication_id of 9004, meaning 4 attempts are stuck at approval waiting

In [41]:
## excluding ineligible campaigns
query = """
SELECT COUNT(*) as eligible_count
FROM communication_log cl
JOIN campaign c ON cl.communication_id = c.id
WHERE cl.merchant_id = 501
  AND c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
  AND c.processing_status = 'processed'
"""
pd.read_sql_query(query, conn)

,eligible_count
0,26


- Joining `communication_log` to `campaign` and filtering out anything not approved/aborted/resumed/stopped + processed drops the count from 30 to 26. The 4 rows removed all belong to campaign 9004 (customers C11–C14) — confirmed by directly querying `communication_id = 9004`. 
- These sends technically happened, but they doesn't count them because the campaign itself was never officially approved.

In [42]:
## checking which customers sms where retried in different campaign

query = """
SELECT communication_id, customer_id, delivery_status, sent_time
FROM communication_log
WHERE communication_id IN (9001, 9002, 9003, 9201, 9202)
ORDER BY communication_id, customer_id
"""
pd.read_sql_query(query, conn)

,communication_id,customer_id,delivery_status,sent_time
0,9001,C1,900,2026-10-03 10:00:00
1,9001,C10,900,2026-10-03 10:00:00
2,9001,C2,1100,2026-10-03 10:00:00
3,9001,C3,1100,2026-10-03 10:00:00
4,9001,C4,900,2026-10-03 10:00:00
5,9001,C5,900,2026-10-03 10:00:00
6,9001,C6,900,2026-10-03 10:00:00
7,9001,C7,900,2026-10-03 10:00:00
8,9001,C8,900,2026-10-03 10:00:00
9,9001,C9,900,2026-10-03 10:00:00


Pulled the raw rows for campaigns 9001/9002/9003/9201/9202 side by side.
Pattern spotted:
- Customer C2 shows up in both 9001 (failed) and 9002 (delivered) — same person, 2 rows.
- Customer C3 shows up in 9001 (failed), 9002 (failed), AND 9003 (delivered) — same person, 3 rows.
- Customer D1 shows up in both 9201 (failed) and 9202 (delivered) — same person, 2 rows.

These aren't 7 separate people being reached — it's 3 people where attempts were made multiple times

In [ ]:
### mapping each campaign to its root id using a recursive chain function

query = """
WITH RECURSIVE chain(campaign_id, root_id) AS (
    SELECT id, id
    FROM campaign
    WHERE merchant_id = 501
    AND parent_id IS NULL
    AND creation_status IN ('approved','aborted','resumed','stopped')
    AND processing_status = 'processed'
    UNION ALL
    SELECT c.id, chain.root_id
    FROM campaign c
    JOIN chain ON c.parent_id = chain.campaign_id
    WHERE c.creation_status IN ('approved','aborted','resumed','stopped')
      AND c.processing_status = 'processed'
)
SELECT * FROM chain ORDER BY root_id, campaign_id
"""
pd.read_sql_query(query, conn)

,campaign_id,root_id
0,9001,9001
1,9002,9001
2,9003,9001
3,9101,9101
4,9201,9201
5,9202,9201


A WITH RECURSIVE query runs in two phases, repeatedly:

- Base case (runs once): Start with campaigns that have parent_id IS NULL. Those are 9001, 9101, 9201 — the true roots. For each, output (campaign_id, root_id) where both columns equal itself: (9001, 9001), (9101, 9101), (9201, 9201). Think of this as "seed" rows — each of these campaigns is its own root, trivially.
- Recursive case (runs over and over, feeding on its own previous output): Find any campaign c whose parent_id matches a campaign_id I already have in my growing table chain. If found, add a new row: (c.id, chain.root_id). Menaing, the child gets tagged with the same root its parent already had.

In [ ]:
query = """
WITH RECURSIVE chain(campaign_id, root_id) AS (
    SELECT id, id
    FROM campaign
    WHERE merchant_id = 501
    AND parent_id IS NULL
    AND creation_status IN ('approved','aborted','resumed','stopped')
    AND processing_status = 'processed'
    UNION ALL
    SELECT c.id, chain.root_id
    FROM campaign c 
    JOIN chain ON c.parent_id = chain.campaign_id
    WHERE c.creation_status IN ('approved','aborted','resumed','stopped')
      AND c.processing_status = 'processed'
)
SELECT cl.id, cl.communication_id, ch.root_id, cl.customer_id, cl.delivery_status
FROM communication_log cl
JOIN chain ch ON cl.communication_id = ch.campaign_id
WHERE cl.merchant_id = 501
ORDER BY ch.root_id, cl.customer_id
"""
pd.read_sql_query(query, conn)

,id,communication_id,root_id,customer_id,delivery_status
0,1,9001,9001,C1,900
1,13,9001,9001,C10,900
2,2,9001,9001,C2,1100
3,3,9002,9001,C2,900
4,4,9001,9001,C3,1100
5,5,9002,9001,C3,1100
6,6,9003,9001,C3,900
7,7,9001,9001,C4,900
8,8,9001,9001,C5,900
9,9,9001,9001,C6,900


- Tagging every log row with its family root
- joined the comm_log with the chain map. This specifies the campaign the sms belongs to and which root campaign it is part of
- we get 26 total rows matching the count we had were we had all approved once except 9004

In [31]:
query = """
WITH RECURSIVE chain(campaign_id, root_id) AS (
    SELECT id, id
    FROM campaign
    WHERE merchant_id = 501
    AND parent_id IS NULL
    AND creation_status IN ('approved','aborted','resumed','stopped')
    AND processing_status = 'processed'
    UNION ALL
    SELECT c.id, chain.root_id
    FROM campaign c JOIN chain ON c.parent_id = chain.campaign_id
    WHERE c.creation_status IN ('approved','aborted','resumed','stopped')
      AND c.processing_status = 'processed'
),
mapped AS (
    SELECT cl.customer_id, ch.root_id
    FROM communication_log cl
    JOIN chain ch ON cl.communication_id = ch.campaign_id
    WHERE cl.merchant_id = 501
)
SELECT root_id, COUNT(*) as total_rows, COUNT(DISTINCT customer_id) as distinct_customers
FROM mapped GROUP BY root_id
"""
pd.read_sql_query(query, conn)

,root_id,total_rows,distinct_customers
0,9001,13,10
1,9101,7,6
2,9201,6,5


total > distinct — proof that customers are being counted more than once within those chains

In [32]:
query = "SELECT parent_id AS root_id, COUNT(*) AS n_children FROM campaign WHERE parent_id IS NOT NULL GROUP BY parent_id"
pd.read_sql_query(query, conn)

,root_id,n_children
0,9001,2
1,9002,1
2,9201,1


Identifying which roots are actual retry chains
- Checked `campaign.parent_id` to see which root campaigns have other campaigns pointing at them. Only 9001 and 9201 have children (9002/9003, and 9202 respectively) — meaning they're real retry chains and should be deduplicated by customer.

- 9101 does NOT appear in this list — nothing points at it, and it points at nothing. It's a standalone campaign. Per the data dictionary, standalone campaigns should NOT be deduplicated — a customer targeted twice under a standalone campaign is two legitimate separate events, not a retry. So for 9101 we keep total_rows = 7, not distinct_customers = 6.

In [33]:
query = """
WITH RECURSIVE chain(campaign_id, root_id) AS (
    SELECT id, id
    FROM campaign
    WHERE merchant_id = 501
      AND parent_id IS NULL
      AND creation_status IN ('approved','aborted','resumed','stopped')
      AND processing_status = 'processed'
    UNION ALL
    SELECT c.id, chain.root_id
    FROM campaign c
    JOIN chain ON c.parent_id = chain.campaign_id
    WHERE c.creation_status IN ('approved','aborted','resumed','stopped')
      AND c.processing_status = 'processed'
),
mapped AS (
    SELECT cl.id AS log_id, cl.customer_id, ch.root_id
    FROM communication_log cl
    JOIN chain ch ON cl.communication_id = ch.campaign_id
    WHERE cl.merchant_id = 501
),
chain_roots AS (
    SELECT DISTINCT parent_id AS root_id FROM campaign WHERE parent_id IS NOT NULL
),
per_root AS (
    SELECT
      m.root_id,
      CASE WHEN cr.root_id IS NOT NULL
           THEN COUNT(DISTINCT m.customer_id)   -- real chain: dedupe per customer
           ELSE COUNT(*)                         -- standalone: every row counts
      END AS qualifying_count
    FROM mapped m
    LEFT JOIN chain_roots cr ON m.root_id = cr.root_id
    GROUP BY m.root_id, cr.root_id
)
SELECT SUM(qualifying_count) AS target_base
FROM per_root
"""
pd.read_sql_query(query, conn)

,target_base
0,22


- 9001 (real chain) → use distinct customers = 10
- 9201 (real chain) → use distinct customers = 5
- 9101 (standalone) → use total rows = 7

10 + 5 + 7 = 22